# TraceGen — recompute the paper table from raw results (Layer 1)

This notebook rebuilds **Table `tab:exp:tg`** of the paper (TraceGen 3D trajectory errors on Droid and EpicKitchen)
from the released per-sample predictions. No GPU and no model are needed.

What it does:

1. Loads, for every test sample, the model's predicted 3D trajectory of 400 keypoints over 33 time steps and the ground truth.
2. Applies the TraceGen authors' test filter (which samples count).
3. Computes MSE, MAE and endpoint MSE (all ×100) exactly as the authors' code does.
4. Prints the table.

Data location: `release_data/tracegen/L1_results/` at the repository root (set the environment variable `WMM_DATA`
to use another `release_data` folder).

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tg_l1

ROOT = tg_l1.l1_root()
print("L1 results:", ROOT)
for env in tg_l1.ENVS:
    print(env, "ground-truth chunks:", len(list((ROOT / "gt" / env).glob("chunk_*.npz"))))

## The metric in plain words

For each kept test sample we have `pred` and `gt` of shape `(33, 400, 3)`: 33 time steps (the start frame plus 32 future steps),
400 keypoints, and 3 coordinates (x, y in normalized image coordinates, z = depth). A boolean `mask` of the same shape says which
ground-truth values exist (frames after the video ends are invalid).

* **MSE** — average squared error over all valid coordinates of all kept samples.
* **MAE** — average absolute error over the same coordinates.
* **Endpoint** — for each keypoint take its last valid time step, then average the squared error of those end positions.

All three are multiplied by 100. Lower is better. Averages are pooled over all coordinates of all kept samples
(not averaged per sample first), as in the authors' code, so the notebook keeps per-sample *sums* and divides at the end.

In [ ]:
per_sample = {}
for (row, env), run in tg_l1.RUNS.items():
    per_sample[(row, env)] = tg_l1.run_per_sample(run, env)
    print(f"{row:22s} {env:5s} {run}")

example = per_sample[("per-dataset (P=128)", "droid")]
example.head()

Each row above is one test sample: `sse`/`sae` are its summed squared/absolute errors over valid coordinates, `n_coord` the number
of valid coordinates, `ep_sse`/`ep_coord` the same for the endpoints, and `kept` says whether the authors' filter keeps it.

In [ ]:
results = {}
for (row, env), df in per_sample.items():
    agg = tg_l1.aggregate(df)
    assert agg["n_valid"] == tg_l1.N_KEPT[env], (row, env, agg["n_valid"])
    results[(row, env)] = agg

table = pd.DataFrame(
    {row: [results[(row, env)][m] for env in tg_l1.ENVS for m in ("MSE", "MAE", "Endp.")]
     for row in dict.fromkeys(r for r, _ in tg_l1.RUNS)},
    index=tg_l1.COLUMNS,
).T
quoted = pd.DataFrame({r: tg_l1.PAPER[r] for r in tg_l1.QUOTED}, index=tg_l1.COLUMNS).T
full = pd.concat([quoted, table])
print("TraceGen table, recomputed (x100, lower is better). Kept samples: Droid",
      tg_l1.N_KEPT["droid"], "/", tg_l1.N_TOTAL["droid"], ", EpicKitchen", tg_l1.N_KEPT["epic"], "/", tg_l1.N_TOTAL["epic"])
print(full.to_string(float_format=lambda v: f"{v:.4f}"))
for r, src in tg_l1.QUOTED.items():
    print(f"\n'{r}' is quoted from {src}.")

## A look at the per-sample errors

Distribution of the per-sample MAE (×100) for the best model on each dataset. A few hard samples have much larger errors
than the typical one; the table value is the pooled average.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, env, name in zip(axes, tg_l1.ENVS, ("Droid", "EpicKitchen")):
    for row in ("per-dataset (P=128)", "Joint (P=128)"):
        k = per_sample[(row, env)].query("kept")
        ax.hist(100 * k["sae"] / k["n_coord"], bins=np.linspace(0, 10, 81), histtype="step", label=row)
    ax.set_title(name)
    ax.set_xlabel("per-sample MAE x100")
    ax.set_ylabel("samples")
    ax.legend()
plt.tight_layout()
plt.show()